In [1]:
!pip install joblib rapidfuzz xgboost

In [ ]:
import pandas as pd
import numpy as np
import jellyfish
from rapidfuzz import distance
import re
import gc
import time
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
import joblib
import json
import warnings

warnings.filterwarnings('ignore')

# Force Linux to use 'fork' to instantly share RAM across all 192 processes
try:
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass

# ==============================================================================
# 1. LOAD GLOBAL DICTIONARIES INTO RAM
# ==============================================================================
print("Loading Text Data into Global RAM...")
cols = ['entity_id', 'clean_name', 'Trans_name', 'norm_address', 'country']
s1 = pd.read_csv("../ML2026/cleaned_dataset/source1.tsv", sep='\t', usecols=cols).fillna('')
s2 = pd.read_csv("../ML2026/cleaned_dataset/source2.tsv", sep='\t', usecols=cols).fillna('')
s3 = pd.read_csv("../ML2026/cleaned_dataset/source3.tsv", sep='\t', usecols=cols).fillna('')

target_df = pd.concat([s2, s3], ignore_index=True)
del s2, s3
gc.collect()

print("Building Global O(1) Hash Maps...")
# These dictionaries will sit in global memory and be read concurrently by 192 cores
global_s1_dict = s1.set_index('entity_id').to_dict(orient='index')
global_tgt_dict = target_df.set_index('entity_id').to_dict(orient='index')

del s1, target_df
gc.collect()

# ==============================================================================
# 2. IN-MEMORY WORKER FUNCTION
# ==============================================================================
def process_memory_chunk(df_chunk):
    """
    Receives a small DataFrame chunk (just IDs), reads from global dicts, 
    computes features, and returns the numerical DataFrame.
    """
    n_rows = len(df_chunk)
    
    jaro_name = np.zeros(n_rows, dtype=np.float32)
    trans_jaro = np.zeros(n_rows, dtype=np.float32)
    lev_address = np.zeros(n_rows, dtype=np.float32)
    country_match = np.zeros(n_rows, dtype=np.int8)
    len_diff_name = np.zeros(n_rows, dtype=np.int16)
    exact_name_match = np.zeros(n_rows, dtype=np.int8)
    name_jaccard = np.zeros(n_rows, dtype=np.float32)
    addr_jaccard = np.zeros(n_rows, dtype=np.float32)
    addr_missing = np.zeros(n_rows, dtype=np.int8)
    addr_digit_match = np.zeros(n_rows, dtype=np.int8)

    digit_pattern = re.compile(r'\d+')
    s1_ids = df_chunk['s1_entity_id'].values
    tgt_ids = df_chunk['target_entity_id'].values

    for i in range(n_rows):
        # O(1) lookups from shared global memory
        s1_data = global_s1_dict.get(s1_ids[i], {})
        tgt_data = global_tgt_dict.get(tgt_ids[i], {})

        s1_n, tgt_n = s1_data.get('clean_name', ''), tgt_data.get('clean_name', '')
        t1, t2 = s1_data.get('clean_name', ''), tgt_data.get('Trans_name', '')
        addr1, addr2 = s1_data.get('norm_address', ''), tgt_data.get('norm_address', '')
        c1, c2 = s1_data.get('country', ''), tgt_data.get('country', '')
        
        exact_name_match[i] = 1 if (s1_n == tgt_n and s1_n != "") else 0
        
        # String Math
        jaro_name[i] = distance.JaroWinkler.normalized_similarity(s1_n, tgt_n)
        trans_jaro[i] = distance.JaroWinkler.normalized_similarity(t1, t2)
        len_diff_name[i] = abs(len(s1_n) - len(tgt_n))
        
        s1_toks, tgt_toks = set(s1_n.split()), set(tgt_n.split())
        if s1_toks and tgt_toks:
            name_jaccard[i] = len(s1_toks & tgt_toks) / len(s1_toks | tgt_toks)
            
        addr_missing[i] = 1 if (addr1 == "" or addr2 == "") else 0
        
        max_l = max(len(addr1), len(addr2))
        if max_l > 0:
            lev_address[i] = 1.0 - (distance.Levenshtein.distance(addr1, addr2) / max_l)
            
        a1_toks, a2_toks = set(addr1.split()), set(addr2.split())
        if a1_toks and a2_toks:
            addr_jaccard[i] = len(a1_toks & a2_toks) / len(a1_toks | a2_toks)
            
        nums1, nums2 = set(digit_pattern.findall(addr1)), set(digit_pattern.findall(addr2))
        if nums1 and nums2:
            addr_digit_match[i] = 1 if (nums1 & nums2) else -1
            
        if c1 == c2 and c1 != "":
            country_match[i] = 1

    # Attach computed arrays
    df_chunk['jaro_name'] = jaro_name
    df_chunk['trans_jaro'] = trans_jaro
    df_chunk['lev_address'] = lev_address
    df_chunk['country_match'] = country_match
    df_chunk['len_diff_name'] = len_diff_name
    df_chunk['exact_name_match'] = exact_name_match
    df_chunk['name_jaccard'] = name_jaccard
    df_chunk['addr_jaccard'] = addr_jaccard
    df_chunk['addr_missing'] = addr_missing
    df_chunk['addr_digit_match'] = addr_digit_match
    
    return df_chunk

# ==============================================================================
# 3. 192-CORE PARALLEL EXECUTION
# ==============================================================================
if __name__ == '__main__':
    print("Loading Candidates into RAM...")
    candidates = pd.read_csv("../method4_lgbm/tfidf_candidate_pairs.csv")
    
    print("Executing 192-Core In-Memory Feature Engineering...")
    start_feat = time.time()
    
    # Split into small chunks to guarantee even load balancing across 192 cores
    df_chunks = np.array_split(candidates, 192 * 4)
    
    with ProcessPoolExecutor(max_workers=192) as executor:
        df = pd.concat(list(executor.map(process_memory_chunk, df_chunks)), ignore_index=True)

    del df_chunks
    gc.collect()
    print(f"Feature computation completed in {time.time() - start_feat:.2f} seconds.")

    # Free the global dictionaries before XGBoost takes over
    del global_s1_dict, global_tgt_dict
    gc.collect()

    # ==============================================================================
    # 4. LABELING & GROUPED SPLIT
    # ==============================================================================
    print("Loading Ground Truth and Labeling...")
    ground_truth = pd.read_csv("../ML2026/student_resource/dataset/train/train_ground_truth.tsv", sep="\t")
    ground_truth['matched_entity_ids'] = ground_truth['matched_entity_ids'].fillna('').str.split(',')
    gt_exploded = ground_truth.explode('matched_entity_ids')
    gt_exploded = gt_exploded[gt_exploded['matched_entity_ids'] != '']
    true_pairs = set(zip(gt_exploded['source1_entity_id'], gt_exploded['matched_entity_ids']))

    candidate_tuples = list(zip(df['s1_entity_id'], df['target_entity_id']))
    df['label'] = np.array([1 if pair in true_pairs else 0 for pair in candidate_tuples], dtype=np.int8)
    
    del true_pairs, candidate_tuples, ground_truth, gt_exploded
    gc.collect()

    features = [
        'jaro_name', 'trans_jaro', 'name_jaccard', 'exact_name_match', 'len_diff_name',
        'lev_address', 'addr_jaccard', 'addr_missing', 'addr_digit_match', 'country_match'
    ]

    print("Splitting Data (Grouped by S1 Entity ID)...")
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(gss.split(df, groups=df['s1_entity_id']))

    X_train, y_train = df.iloc[train_idx][features], df.iloc[train_idx]['label']
    X_val, y_val = df.iloc[val_idx][features], df.iloc[val_idx]['label']
    val_data_metadata = df.iloc[val_idx][['s1_entity_id', 'target_entity_id', 'label']].copy()

    # Delete the massive un-split dataframe to keep RAM perfectly clean for XGBoost
    del df
    gc.collect()

    pos_weight = float((y_train == 0).sum() / (y_train == 1).sum()) if (y_train == 1).sum() > 0 else 1.0

    # ==============================================================================
    # 5. XGBOOST TRAINING & CALIBRATION
    # ==============================================================================
    print("\nTraining Precision-Heavy XGBoost Classifier on 192 Cores...")
    xgb_model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.9,
        scale_pos_weight=pos_weight,
        tree_method='hist',           
        n_jobs=192,                   # Saturate all 192 cores instantly
        random_state=42,
        eval_metric='logloss'
    )

    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50
    )

    print("\nCalibrating F0.5 Threshold:")
    val_data_metadata['match_prob'] = xgb_model.predict_proba(X_val)[:, 1]
    
    best_f05, best_thresh = 0, 0.5
    for thresh in np.arange(0.70, 0.98, 0.02):
        pred = (val_data_metadata['match_prob'] >= thresh).astype(int)
        tp = np.sum((pred == 1) & (val_data_metadata['label'] == 1))
        fp = np.sum((pred == 1) & (val_data_metadata['label'] == 0))
        fn = np.sum((pred == 0) & (val_data_metadata['label'] == 1))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f05 = (1.25 * precision * recall) / ((0.25 * precision) + recall) if (precision + recall) > 0 else 0
        
        if f05 > best_f05:
            best_f05, best_thresh = f05, thresh
            
        print(f"Thresh: {thresh:.2f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F0.5: {f05:.4f}")

    print(f"\n--> Optimal F0.5 Threshold: {best_thresh:.2f}")

    joblib.dump(xgb_model, 'xgboost_entity_resolution.pkl')
    with open('threshold_config.json', 'w') as f:
        json.dump({'best_threshold': float(best_thresh)}, f)
    print("Model saved successfully! Ready for Test Inference.")

In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
import jellyfish
from rapidfuzz import distance
import re
import gc
import time
import os
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm
import joblib
import json
from anyascii import anyascii
import warnings

warnings.filterwarnings('ignore')

try:
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass

# ==============================================================================
# 1. EDA & PREPROCESSING DEFINITIONS
# ==============================================================================
RAW_MULTILINGUAL_SEEDS = {
    'private': ['प्राइवेट', 'प्रायव्हेट', 'ప్రైవేట్', 'பிரைவேட்', 'প্রাইভেট', 'પ્રાઇવેટ', 'ಖಾಸಗಿ', 'പ്രൈവറ്റ്', 'ਪ੍ਰਾਈਵੇਟ', 'ପ୍ରାଇଭେଟ', 'پرائیویٹ', 'privé', 'privée'],
    'limited': ['लिमिटेड', 'मर्यादित', 'లిమిటెడ్', 'லிமிடெட்', 'লিমিটেড', 'લિમિટેડ', 'ಲಿಮಿಟೆಡ್', 'ലിമിറ്റഡ്', 'ਲਿਮਿਟੇਡ', 'ଲିମିଟେଡ', 'لمیٹڈ', 'limitée', 'limité'],
    'company': ['कंपनी', 'కంపెనీ', 'கம்பெனி', 'কোম্পানি', 'કંપની', 'ಕಂಪನಿ', 'കമ്പനി', 'ਕੰਪਨੀ', 'କମ୍ପାନୀ', 'کمپنی', 'société', 'compagnie'],
    'corporation': ['कॉर्पोरेशन', 'निगम', 'కార్పొరేషన్', 'கார்ப்பரேஷன்', 'কর্পোরেশন', 'કોર્પોરેશન', 'ಕಾರ್ಪೊರೇಷನ್', 'കോർപ്പറേഷൻ', 'ਕਾਰਪੋਰੇਸ਼ਨ', 'କର୍ପୋରେସନ', 'کارپوریشن'],
    'incorporated': ['इनकॉर्पोरेटेड', 'ఇన్కార్పొరేటెड', 'இன்கார்பரேட்டட்', 'ইনকর্পোরেটেড', 'ઇનકોર્પોરેટેડ', 'ಇನ್ಕಾರ್ಪೊರೇಟೆಡ್', 'ഇൻകോർപ്പറേറ്റഡ്', 'ਇਨਕਾਰਪੋਰੇਟਿਡ', 'ଇନକର୍ପୋରେଟେਡ'],
    'llp': ['एलएलपी', 'ఎల్‌ఎల్‌పి', 'எல்.எல்.பி', 'এলএলপি', 'એલએલપી', 'ಎಲ್ಎಲ್ಪಿ', 'എൽഎൽപി', 'ਐਲਐਲਪੀ', 'ଏଲଏଲପି'],
    'finance': ['फाइनेंस', 'फायनान्स', 'ఫైనాన్స్', 'பைனான்ஸ்', 'ফাইন্যান্স', 'ફાઇનાન્સ', 'ಫೈನಾನ್ಸ್', 'ഫിനാൻസ്', 'ਫਾਈਨਾਂਸ', 'ଫାଇନାନ୍ସ', 'فنانس', 'finance'],
    'bank': ['बैंक', 'बँक', 'బ్యాంక్', 'வங்கி', 'ব্যাঙ্ক', 'બેંક', 'ಬ್ಯಾಂಕ್', 'ബാങ്ക്', 'ਬੈਂਕ', 'ବ୍ୟାଙ୍କ', 'بینک', 'banque'],
    'insurance': ['इंश्योरेंस', 'विमा', 'ఇన్సూరెన్స్', 'இன்சூரன்ஸ்', 'ইন্স্যুরেন্স', 'ઇન્શ્યોરન્સ', 'ಇನ್ಶೂರೆನ್ಸ್', 'ഇൻഷുറൻസ്', 'ഇੰਸ਼ੋਰੈਂਸ', 'ଇନ୍ସୁରାନ୍ସ', 'انشورنس', 'assurance'],
    'investment': ['इन्वेस्टमेंट', 'गुंतवणूक', 'ఇన్వెస్ట్‌మెంట్', 'இன்வெஸ்ட்மென்ட்', 'ইনভেস্টমেন্ট', 'ઇન્વેસ્ટમેન્ટ', 'ಇನ್ವೆಸ್ಟ್ಮೆಂಟ್', 'ഇൻവെസ്റ്റ്മെന്റ്', 'ਇਨਵੈਸਟਮੈਂਟ', 'ଇନଭେଷ୍ଟମେଣ୍ଟ', 'انویسٹمنٹ', 'investissement'],
    'technologies': ['टेक्नोलॉजीज', 'तंत्रज्ञान', 'టెక్నాలజీస్', 'டெக்னாலஜிஸ்', 'টেকনোলজিস', 'ટેકનોલોજીસ', 'ಟೆಕ್ನಾಲಜೀಸ್', 'ടെക്നോളജീസ്', 'ਟੈਕਨਾਲੋਜੀਜ਼', 'ଟେକ୍ନୋଲୋଜିସ', 'ٹیکنالوجیز'],
    'consultancy': ['कंसल्टेंसी', 'పరామर्श', 'కన్సల్టెన్సీ', 'கன்சல்டன்சி', 'কনসালটেন্সি', 'કન્સલ્ટન્સી', 'ಕನ್ಸಲ್ಟೆನ್ಸಿ', 'കൺസൾട്ടൻസി', 'ਕੰਸਲਟੈਂਸੀ', 'କନସଲଟାନ୍ସି', 'کنسلٹنسی', 'consultance', 'conseil'],
    'pharmacy': ['फार्मेसी', 'औषधालय', 'ఫార్మసీ', 'பார்மசி', 'ফার্মেসি', 'ફાર્મસી', 'ಫಾರ್ಮಸಿ', 'ഫാർമസി', 'ਫਾਰਮੇਸੀ', 'ଫାର୍ମାସି', 'فارمیسی', 'pharmacie'],
    'logistics': ['लॉजिस्टिक्स', 'లాజిస్టిక్స్', 'லாஜிஸ்டிக்ஸ்', 'লজিস্টিকস', 'લોજિસ્ટિક્સ', 'ಲಾಜಿಸ್ಟಿಕ್ಸ್', 'ലോജിസ്റ്റിക്സ്', 'ਲੌਜਿਸਟਿਕਸ', 'ଲଜିଷ୍ଟିକ୍ସ', 'لاجسٹکس', 'logistique'],
    'manufacturing': ['मैन्युफैक्चरिंग', 'उत्पादन', 'మాన్యుఫ్యాక్చరింగ్', 'மேனுஃபேக்ச்சரிங்', 'ম্যানুফ্যাকচারিং', 'મેન્યુફેક્ચરિંગ', 'ಮ್ಯಾನುಫ್ಯಾಕ್ಚರಿಂಗ್', 'മാനുഫാക്ചറിംഗ്', 'ਮੈਨੂਫੈਕਚਰਿੰਗ', 'ମାନୁଫାକ୍ଚରିଂ', 'مینوفیکچرنگ', 'fabrication'],
    'group': ['ग्रुप', 'समूह', 'గ్రూప్', 'குரூப்', 'গ্রুপ', 'ગ્રુપ', 'ಗ್ರೂಪ್', 'ഗ്രൂപ്പ്', 'ਗਰੁੱਪ', 'ଗ୍ରୁପ', 'گروپ', 'groupe'],
    'associates': ['एसोसिएट्स', 'असोसिएट्स', 'అసోసియేట్స్', 'அசோசியேட்ஸ்', 'অ্যাসোসিয়েটস', 'એસોસિએટ્સ', 'ಅಸೋಸಿಯೇಟ್ಸ್', 'അസോസിയേറ്റ്സ്', 'ਐਸੋਸੀਏਟਸ', 'ଆସୋସିଏଟ୍ସ', 'ایسوسی ایٹس', 'associés'],
    'enterprises': ['एंटरप्राइजेज', 'उद्यम', 'ఎంటర్‌ప్రైజెస్', 'என்டர்பிரைசஸ்', 'এন্টারপ্রাইজ', 'એન્ટરપ્રાઇઝિસ', 'ಎಂಟರ್‌ಪ್ರೈಸಸ್', 'എന്റർപ്രൈസಸ್', 'ਐਂਟਰਪ੍ਰਾਈਜ਼ਿਜ਼', 'ଏଣ୍ଟରପ୍ରାଇଜେସ', 'انٹرپرائزز', 'entreprises'],
    'services': ['सर्विसेज', 'सेवा', 'సర్వీసెస్', 'சர்வீசஸ்', 'সার্ভিসেস', 'સર્વિસિસ', 'ಸರ್ವಿಸಸ್', 'സർവീസസ്', 'ਸਰਵਿਸਿਜ਼', 'ସର୍ଭିସେସ', 'سروسز', 'services'],
    'solutions': ['सॉल्यूशंस', 'సొల్యూషన్स', 'சொல்யூஷன்ஸ்', 'সলিউশন', 'સોલ્યુશન્સ', 'ಸಲ್ಯೂಷನ್ಸ್', 'സൊല്യൂഷൻസ്', 'ਸਲਿਊਸ਼ਨਜ਼', 'ସଲ୍ୟୁସନ', 'سلوشنز'],
    'industries': ['इंडस्ट्रीज', 'उद्योग', 'ఇండస్ట్రీస్', 'இண்டஸ்ட்ரீஸ்', 'ইন্ডাস্ট্রিজ', 'ઇન્ડસ્ટ્રીઝ', 'ಇಂಡಸ್ಟ್ರೀಸ್', 'ഇൻഡസ്ട്രീസ്', 'ਇੰਡਸਟਰੀਜ਼', 'ଇଣ୍ଡଷ୍ଟ୍ରିଜ', 'انڈسٹریز'],
    'global': ['ग्लोबल', 'जागतिक', 'గ్లోబల్', 'குளோபல்', 'গ্লোবাল', 'ગ્લોબલ', 'ಗ್ಲೋಬಲ್', 'ഗ്ലോബൽ', 'ਗਲੋਬਲ', 'ଗ୍ଲୋବାଲ', 'گلوبل', 'mondial'],
    'ventures': ['वेंचर्स', 'వెంచర్స్', 'வென்சர்ஸ்', 'ভেঞ্চারস', 'વેન્ચર્સ', 'ವೆಂಚರ್ಸ್', 'വെഞ്ചേഴ്സ്', 'ਵੈਂਚਰਸ', 'ଭେଞ୍ଚର୍ସ', 'وینچرز']
}

def build_compiled_phonetic_dictionary(seed_dict):
    compiled_patterns = {}
    for canonical_english, native_terms in seed_dict.items():
        ascii_variants = set()
        for term in native_terms:
            ascii_repr = anyascii(term).lower().strip()
            ascii_repr = re.sub(r'[^\w\s]', '', ascii_repr)
            if ascii_repr:
                ascii_variants.add(ascii_repr)
        sorted_variants = sorted(list(ascii_variants), key=len, reverse=True)
        pattern_str = r'\b(' + '|'.join(map(re.escape, sorted_variants)) + r')\b'
        compiled_patterns[re.compile(pattern_str)] = canonical_english
    return compiled_patterns

PHONETIC_REGEX_MAPPING = build_compiled_phonetic_dictionary(RAW_MULTILINGUAL_SEEDS)

NAME_SUFFIX_MAPPING = {
    re.compile(r'\bcorp\b'): 'corporation',
    re.compile(r'\bpvt\b'): 'private',
    re.compile(r'\bltd\b'): 'limited',
    re.compile(r'\binc\b'): 'incorporated',
    re.compile(r'\bllc\b'): 'limited liability company',
    re.compile(r'\bco\b'): 'company',
    re.compile(r'\bpc\b'): 'professional corporation'
}

ADDR_MAPPING = {
    re.compile(r'\brd\b'): 'road',
    re.compile(r'\bst\b'): 'street',
    re.compile(r'\bave\b'): 'avenue',
    re.compile(r'\bblvd\b'): 'boulevard',
    re.compile(r'\bbldg\b'): 'building',
    re.compile(r'\bapt\b'): 'apartment',
    re.compile(r'\bste\b'): 'suite',
    re.compile(r'\bfl\b'): 'floor',
    re.compile(r'\bno\b'): 'number'
}

RE_LANDMARK = re.compile(r'\b(near|opp|opposite)\s+\w+(?:\s+\w+){0,3}')
RE_NULL = re.compile(r'(\b(na|null|nan|nill)\b)')
RE_PUNCT = re.compile(r'[^\w\s]')

# Pre-compile the company token regex for extreme performance
COMPANY_TOKEN_PATTERN = re.compile(
    r'\b(pvt|private|ltd|limited|llp|llc|inc|incorporated|corp|corporation|co|company|lp|associates|group|liability)\b', 
    flags=re.IGNORECASE
)

def std_bus_name(name):
    if pd.isna(name):
        return ""
    name = str(name).lower().replace("&", " and ")
    name = RE_PUNCT.sub('', name)
    for pattern, replacement in NAME_SUFFIX_MAPPING.items():
        name = pattern.sub(replacement, name)
    return " ".join(name.split())

def norm_add(address):
    if pd.isna(address):
        return ""
    address = str(address).lower()
    address = RE_PUNCT.sub('', address)
    for pattern, replacement in ADDR_MAPPING.items():
        address = pattern.sub(replacement, address)
    address = RE_LANDMARK.sub(' ', address)
    address = RE_NULL.sub('', address)
    return " ".join(address.split())

def clean_name_data(norm_name_str, norm_addr_str):
    x_tokens = [tok for tok in norm_name_str.split() if not tok.isdigit()]
    y_tokens = set(norm_addr_str.split())
    final_tokens = [token for token in x_tokens if token not in y_tokens]
    return " ".join(final_tokens)

def normalize_text_pipeline(text):
    if pd.isna(text) or not text:
        return ""
    text_ascii = anyascii(str(text)).lower()
    text_ascii = RE_PUNCT.sub(' ', text_ascii)
    for pattern, canonical_term in PHONETIC_REGEX_MAPPING.items():
        text_ascii = pattern.sub(canonical_term, text_ascii)
    return " ".join(text_ascii.split())

def process_dataframe_chunk(df_chunk):
    df_chunk = df_chunk.copy()
    norm_names = [std_bus_name(x) for x in df_chunk['business_name'].values]
    norm_addrs = [norm_add(x) for x in df_chunk['business_address'].values]
    
    # Extract clean name by stripping address tokens
    clean_names = [clean_name_data(n, a) for n, a in zip(norm_names, norm_addrs)]
    
    # Strip generic legal tokens using the pre-compiled regex
    clean_names = [COMPANY_TOKEN_PATTERN.sub('', cn) for cn in clean_names]
    
    # Final space normalization
    clean_names = [" ".join(cn.split()) for cn in clean_names]
    
    df_chunk['norm_name'] = norm_names
    df_chunk['norm_address'] = norm_addrs
    df_chunk['clean_name'] = clean_names
    df_chunk['Trans_name'] = [normalize_text_pipeline(cn) for cn in clean_names]
    return df_chunk

def parallel_preprocess(df, num_workers=192):
    chunks = np.array_split(df, num_workers * 2)
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        processed_chunks = list(tqdm(executor.map(process_dataframe_chunk, chunks), total=len(chunks), desc="Preprocessing Data"))
    return pd.concat(processed_chunks, ignore_index=True)

# ==============================================================================
# 2. FEATURE ENGINEERING WORKER
# ==============================================================================
def process_memory_chunk(df_chunk):
    n_rows = len(df_chunk)
    
    jaro_name = np.zeros(n_rows, dtype=np.float32)
    trans_jaro = np.zeros(n_rows, dtype=np.float32)
    lev_address = np.zeros(n_rows, dtype=np.float32)
    country_match = np.zeros(n_rows, dtype=np.int8)
    len_diff_name = np.zeros(n_rows, dtype=np.int16)
    exact_name_match = np.zeros(n_rows, dtype=np.int8)
    name_jaccard = np.zeros(n_rows, dtype=np.float32)
    addr_jaccard = np.zeros(n_rows, dtype=np.float32)
    addr_missing = np.zeros(n_rows, dtype=np.int8)
    addr_digit_match = np.zeros(n_rows, dtype=np.int8)

    digit_pattern = re.compile(r'\d+')
    s1_ids = df_chunk['s1_entity_id'].values
    tgt_ids = df_chunk['target_entity_id'].values

    for i in range(n_rows):
        s1_data = global_s1_dict.get(s1_ids[i], {})
        tgt_data = global_tgt_dict.get(tgt_ids[i], {})

        s1_n, tgt_n = s1_data.get('clean_name', ''), tgt_data.get('clean_name', '')
        t1, t2 = s1_data.get('clean_name', ''), tgt_data.get('Trans_name', '')
        addr1, addr2 = s1_data.get('norm_address', ''), tgt_data.get('norm_address', '')
        c1, c2 = s1_data.get('country', ''), tgt_data.get('country', '')
        
        exact_name_match[i] = 1 if (s1_n == tgt_n and s1_n != "") else 0
        jaro_name[i] = distance.JaroWinkler.normalized_similarity(s1_n, tgt_n)
        trans_jaro[i] = distance.JaroWinkler.normalized_similarity(t1, t2)
        len_diff_name[i] = abs(len(s1_n) - len(tgt_n))
        
        s1_toks, tgt_toks = set(s1_n.split()), set(tgt_n.split())
        if s1_toks and tgt_toks:
            name_jaccard[i] = len(s1_toks & tgt_toks) / len(s1_toks | tgt_toks)
            
        addr_missing[i] = 1 if (addr1 == "" or addr2 == "") else 0
        
        max_l = max(len(addr1), len(addr2))
        if max_l > 0:
            lev_address[i] = 1.0 - (distance.Levenshtein.distance(addr1, addr2) / max_l)
            
        a1_toks, a2_toks = set(addr1.split()), set(addr2.split())
        if a1_toks and a2_toks:
            addr_jaccard[i] = len(a1_toks & a2_toks) / len(a1_toks | a2_toks)
            
        nums1, nums2 = set(digit_pattern.findall(addr1)), set(digit_pattern.findall(addr2))
        if nums1 and nums2:
            addr_digit_match[i] = 1 if (nums1 & nums2) else -1
            
        if c1 == c2 and c1 != "":
            country_match[i] = 1

    df_chunk['jaro_name'] = jaro_name
    df_chunk['trans_jaro'] = trans_jaro
    df_chunk['lev_address'] = lev_address
    df_chunk['country_match'] = country_match
    df_chunk['len_diff_name'] = len_diff_name
    df_chunk['exact_name_match'] = exact_name_match
    df_chunk['name_jaccard'] = name_jaccard
    df_chunk['addr_jaccard'] = addr_jaccard
    df_chunk['addr_missing'] = addr_missing
    df_chunk['addr_digit_match'] = addr_digit_match
    
    return df_chunk

# ==============================================================================
# 3. PIPELINE EXECUTION
# ==============================================================================
if __name__ == '__main__':
    print("Loading XGBoost Model and Threshold...")
    xgb_model = joblib.load('xgboost_entity_resolution.pkl')
    with open('threshold_config.json', 'r') as f:
        best_thresh = json.load(f)['best_threshold']

    print("Reading Raw TSV Test Files...")
    test_s1 = pd.read_csv("../ML2026/student_resource/dataset/test/test_source1.tsv", sep='\t')
    test_s2 = pd.read_csv("../ML2026/student_resource/dataset/test/test_source2.tsv", sep='\t')
    test_s3 = pd.read_csv("../ML2026/student_resource/dataset/test/test_source3.tsv", sep='\t')
    
    test_s1_ids_full = test_s1['entity_id'].values

    print("\n[Phase 1] Executing 192-Core Parallel EDA Preprocessing (With Suffix Removal)...")
    start_eda = time.time()
    test_s1 = parallel_preprocess(test_s1, num_workers=192)
    test_s2 = parallel_preprocess(test_s2, num_workers=192)
    test_s3 = parallel_preprocess(test_s3, num_workers=192)
    print(f"Preprocessing completed in {time.time() - start_eda:.2f} seconds.")

    print("\n[Phase 2] Generating Candidates via TF-IDF + Sparse Dot Top-N...")
    start_tfidf = time.time()
    cols = ['entity_id', 'clean_name', 'Trans_name', 'norm_address', 'country']
    target_df = pd.concat([test_s2[cols], test_s3[cols]], ignore_index=True)

    def build_search_corpus(df):
        return (df['clean_name'].fillna('') + " " + df['Trans_name'].fillna('') + " " + df['norm_address'].fillna('')).str.lower()

    vectorizer = TfidfVectorizer(
        analyzer='char_wb', 
        ngram_range=(3, 3), 
        min_df=5,               
        max_df=0.01,             
        dtype=np.float32         
    )
    
    target_corpus = build_search_corpus(target_df)
    test_corpus = build_search_corpus(test_s1)
    
    print("Fitting vocabulary dictionary on test targets...")
    vectorizer.fit(target_corpus)
    
    print("Transforming text to mathematical matrices...")
    target_matrix = vectorizer.transform(target_corpus)
    test_matrix = vectorizer.transform(test_corpus)
    target_matrix_T = target_matrix.transpose()
    
    del target_corpus, test_corpus
    gc.collect()

    print("Running 192-Core Sparse Dot Product...")
    chunk_size = 50000
    test_ids = test_s1['entity_id'].values
    target_ids = target_df['entity_id'].values
    
    test_cand_s1 = []
    test_cand_tgt = []
    
    for i in tqdm(range(0, test_matrix.shape[0], chunk_size), desc="Matrix Multiplication"):
        end = min(i + chunk_size, test_matrix.shape[0])
        chunk_matches = sp_matmul_topn(
            test_matrix[i:end], 
            target_matrix_T, 
            top_n=20, 
            threshold=0.60, 
            n_threads=192
        )
        non_zeros = chunk_matches.nonzero()
        test_cand_s1.extend(test_ids[non_zeros[0] + i])
        test_cand_tgt.extend(target_ids[non_zeros[1]])
    
    candidates = pd.DataFrame({
        's1_entity_id': test_cand_s1,
        'target_entity_id': test_cand_tgt
    })
    
    del test_matrix, target_matrix, target_matrix_T, test_cand_s1, test_cand_tgt
    gc.collect()
    print(f"Candidate generation completed in {time.time() - start_tfidf:.2f} seconds.")

    print("\n[Phase 2.5] Formatting and Exporting Raw Candidate Pairs...")
    os.makedirs("output", exist_ok=True)
    
    # Format candidates for the competition validator (Grouped by S1, comma-separated Targets)
    raw_candidates = candidates.groupby('s1_entity_id')['target_entity_id'].apply(
        lambda x: ','.join(list(dict.fromkeys(x)))
    ).reset_index()
    raw_candidates.rename(columns={'s1_entity_id': 'source1_entity_id', 'target_entity_id': 'candidate_entity_ids'}, inplace=True)

    all_test_ids = pd.DataFrame({'source1_entity_id': test_s1_ids_full})
    candidate_submission = all_test_ids.merge(raw_candidates, on='source1_entity_id', how='left')
    candidate_submission['candidate_entity_ids'] = candidate_submission['candidate_entity_ids'].fillna('')
    
    candidate_output_file = "output/candidate_pairs.tsv"
    candidate_submission.to_csv(candidate_output_file, sep='\t', index=False)
    print(f"Exported Raw Candidates: {candidate_output_file} ({len(candidate_submission):,} rows)")
    del raw_candidates, candidate_submission
    gc.collect()

    print("\n[Phase 3] Engineering 10 XGBoost Features (192 Cores In-Memory)...")
    start_feat = time.time()
    
    global_s1_dict = test_s1.set_index('entity_id').to_dict(orient='index')
    global_tgt_dict = target_df.set_index('entity_id').to_dict(orient='index')
    
    del test_s1, target_df
    gc.collect()
    
    df_chunks = np.array_split(candidates, 192 * 4)
    with ProcessPoolExecutor(max_workers=192) as executor:
        results = list(tqdm(executor.map(process_memory_chunk, df_chunks), total=len(df_chunks), desc="Engineering Features"))
        df = pd.concat(results, ignore_index=True)

    del df_chunks, results, global_s1_dict, global_tgt_dict
    gc.collect()
    print(f"Features engineered in {time.time() - start_feat:.2f} seconds.")

    print("\n[Phase 4] Running XGBoost Inference & Formatting Final Output...")
    features = [
        'jaro_name', 'trans_jaro', 'name_jaccard', 'exact_name_match', 'len_diff_name',
        'lev_address', 'addr_jaccard', 'addr_missing', 'addr_digit_match', 'country_match'
    ]

    df['match_prob'] = xgb_model.predict_proba(df[features])[:, 1]
    df['is_match'] = (df['match_prob'] >= best_thresh).astype(int)

    final_matches = df[df['is_match'] == 1]
    
    submission = final_matches.groupby('s1_entity_id')['target_entity_id'].apply(
        lambda x: ','.join(list(dict.fromkeys(x))) 
    ).reset_index()
    
    submission.rename(columns={'s1_entity_id': 'source1_entity_id', 'target_entity_id': 'matched_entity_ids'}, inplace=True)

    submission_final = all_test_ids.merge(submission, on='source1_entity_id', how='left')
    submission_final['matched_entity_ids'] = submission_final['matched_entity_ids'].fillna('')
    
    final_output_file = "output/matching_results.tsv"
    submission_final.to_csv(final_output_file, sep='\t', index=False)
    
    print(f"\nSUCCESS: Inference Complete.")
    print(f"Generated Final Predictions: {final_output_file} ({len(submission_final):,} rows)")

Loading XGBoost Model and Threshold...
Reading Raw TSV Test Files...

[Phase 1] Executing 192-Core Parallel EDA Preprocessing (With Suffix Removal)...


Preprocessing Data:   0%|          | 0/384 [00:00<?, ?it/s]

Preprocessing Data:   0%|          | 0/384 [00:00<?, ?it/s]

Preprocessing Data:   0%|          | 0/384 [00:00<?, ?it/s]

Preprocessing completed in 37.11 seconds.

[Phase 2] Generating Candidates via TF-IDF + Sparse Dot Top-N...
Fitting vocabulary dictionary on test targets...
Transforming text to mathematical matrices...
Running 192-Core Sparse Dot Product...


Matrix Multiplication:   0%|          | 0/35 [00:00<?, ?it/s]

Candidate generation completed in 1468.33 seconds.

[Phase 2.5] Formatting and Exporting Raw Candidate Pairs...
Exported Raw Candidates: output/candidate_pairs.tsv (1,732,544 rows)

[Phase 3] Engineering 10 XGBoost Features (192 Cores In-Memory)...


Engineering Features:   0%|          | 0/768 [00:00<?, ?it/s]

Features engineered in 94.51 seconds.

[Phase 4] Running XGBoost Inference & Formatting Final Output...

SUCCESS: Inference Complete.
Generated Final Predictions: output/matching_results.tsv (1,732,544 rows)


In [16]:
import pandas as pd

print("1. Loading and Exploding Ground Truth...")
# Ground truth grouped hota hai, isliye usko split karke explode karna zaroori hai
gt = pd.read_csv("../ML2026/student_resource/dataset/train/train_ground_truth.tsv", sep="\t")
gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('').astype(str).str.split(',')
gt_exploded = gt.explode('matched_entity_ids')

# Invisible spaces ko strip karein taaki mismatch na ho
gt_exploded['source1_entity_id'] = gt_exploded['source1_entity_id'].astype(str).str.strip()
gt_exploded['matched_entity_ids'] = gt_exploded['matched_entity_ids'].astype(str).str.strip()
gt_exploded = gt_exploded[gt_exploded['matched_entity_ids'] != '']

actual_true_pairs = set(zip(gt_exploded['source1_entity_id'], gt_exploded['matched_entity_ids']))
print(f"Total True Positives in Ground Truth: {len(actual_true_pairs):,}")

print("\n2. Loading Raw Candidate Pairs...")
# Yahan apni actual file ka naam dalein (jaise 'tfidf_candidate_pairs.csv' ya 'strict_s1_candidate_pairs.csv')
candidates = pd.read_csv("tfidf_candidate_pairs.csv") 

# Check if tfidf_similarity exists, drop it if it does as we only need IDs for set intersection
if 'tfidf_similarity' in candidates.columns:
    candidates = candidates[['s1_entity_id', 'target_entity_id']]

# Invisible spaces ko strip karein
candidates['s1_entity_id'] = candidates['s1_entity_id'].astype(str).str.strip()
candidates['target_entity_id'] = candidates['target_entity_id'].astype(str).str.strip()

generated_pairs = set(zip(candidates['s1_entity_id'], candidates['target_entity_id']))
print(f"Total Candidate Pairs Generated: {len(generated_pairs):,}")

print("\n3. Calculating Intersection...")
captured_true_pairs = actual_true_pairs.intersection(generated_pairs)
total_captured = len(captured_true_pairs)
total_actual = len(actual_true_pairs)

blocking_recall = total_captured / total_actual if total_actual > 0 else 0
blocking_precision = total_captured / len(generated_pairs) if len(generated_pairs) > 0 else 0

print("\n" + "="*45)
print("CORRECTED BLOCKING METRICS")
print("="*45)
print(f"True Matches Captured : {total_captured:,} / {total_actual:,}")
print(f"Missed Matches (FN)   : {total_actual - total_captured:,}")
print(f"Blocking Recall       : {blocking_recall:.4f} ({blocking_recall*100:.2f}%)")
print(f"Blocking Precision    : {blocking_precision:.4f} ({blocking_precision*100:.2f}%)")
print("="*45)

1. Loading and Exploding Ground Truth...
Total True Positives in Ground Truth: 7,638,365

2. Loading Raw Candidate Pairs...
Total Candidate Pairs Generated: 30,559,502

3. Calculating Intersection...

CORRECTED BLOCKING METRICS
True Matches Captured : 5,606,780 / 7,638,365
Missed Matches (FN)   : 2,031,585
Blocking Recall       : 0.7340 (73.40%)
Blocking Precision    : 0.1835 (18.35%)


In [17]:
candidates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30559502 entries, 0 to 30559501
Data columns (total 2 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   s1_entity_id      object
 1   target_entity_id  object
dtypes: object(2)
memory usage: 466.3+ MB


In [9]:
gt_exploded.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7638365 entries, 0 to 2206820
Data columns (total 2 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   source1_entity_id   object
 1   matched_entity_ids  object
dtypes: object(2)
memory usage: 174.8+ MB
